In [ ]:
from langgraph_sdk import get_client

client = get_client(url="http://localhost:2024")

assistant_id = "2ac0901b-41bf-4d2b-a3b8-9cdee6a1af53"

# 删除所有线程，免得看晕了
threads = await client.threads.search(limit=100)
for t in threads:
    await client.threads.delete(thread_id=t['thread_id'])

thread = await client.threads.create(
    metadata={"__name__": "事件流"}
)
thread_id = thread["thread_id"]
thread_id

In [ ]:
from typing import AsyncIterator

from langgraph_sdk.schema import StreamPart

msg_types: dict[str, str] = {}
announced_subgraphs: set[str] = set()
msg_type_map = {
    'thinking': '思考：',
    'text': '回复：',
    'tool_use': '调用工具：'
}
def print_message(chunk: StreamPart):
    event_type, _, namespace = chunk.event.partition('|')
    if event_type != "messages": 
        return
    if chunk.data[1]['langgraph_node'] != 'model':
        return
    source = namespace or 'root'
    content_blocks = chunk.data[0]['content']
    for block in content_blocks:
        type = block['type']
        if type != msg_types.get(source):
            msg_types[source] = type
            if type not in msg_type_map:
                continue
            title = msg_type_map[type]
            if type == 'tool_use':
                title += block['name']
            if namespace and namespace not in announced_subgraphs:
                announced_subgraphs.add(namespace)
                print(f"\n子代理：\n{title}", flush=True)
            else:
                print(flush=True)
                print(title, flush=True)
        if type not in msg_type_map:
            continue
        if type == 'tool_use':
            continue 
        print(block[type], end='', flush=True)

chunks = []
async def print_stream(stream: AsyncIterator[StreamPart]):
    global chunks
    chunks = []
    msg_types.clear()
    announced_subgraphs.clear()
    result = {}
    async for chunk in stream:
        chunks.append(chunk)
        print_message(chunk)
        if chunk.event == 'values':
            result = chunk.data
    return result


In [ ]:
stream = client.runs.stream(
    thread_id=thread_id,
    assistant_id=assistant_id,
    input={
        "messages": [
            {"role": "user", "content": "写一首夏天的诗"}
        ]
    },
    stream_mode=[
        "values",
        "checkpoints",
        "updates",
        "messages-tuple" 
    ],
    stream_subgraphs=True,
    version="v1",
)

output = await print_stream(stream)

In [ ]:
output

In [ ]:
from langgraph_sdk.schema import Command as SDKCommand

answers = {
    '806d9079dfa52181485476ba6c5ce802': '清新淡雅'
}
answers

In [ ]:
stream = client.runs.stream(
    thread_id=thread_id,
    assistant_id=assistant_id,
    command=SDKCommand(resume=answers),
    stream_mode=[
        "values",
        "checkpoints",
        "updates",
        "messages-tuple" 
    ],
    stream_subgraphs=True,
    version="v1",
)

output = await print_stream(stream)